# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR\u00b2 dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described using a Croissant schema, which is accessible at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

The FAIR^2 dataset contains proteomic data from mass spectrometry analysis of extracellular vesicles isolated from human mast cells.

In [ ]:
# Ensure `mlcroissant` is available
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records via `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"License: {metadata.license}")

## 2. Data Overview
Let's explore the available record sets, their `@id` values, and the structure of the dataset fields.

In [ ]:
# List all record sets and their fields/columns (@id and descriptions)
record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs.name} (id: {rs.id})")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (id: {field.id}) | Type: {field.data_type if hasattr(field, 'data_type') else ''}")
    print()
record_set_ids = [rs.id for rs in record_sets]

## 3. Data Extraction
Select a record set (by `@id`), and load it as a DataFrame. We'll extract the first available record set and inspect its fields.

In [ ]:
# You may list all record set ids for selection:
print("RecordSet IDs:")
for i, rid in enumerate(record_set_ids):
    print(f"[{i}] {rid}")

# For this notebook, we select the first record set
selected_record_set_id = record_set_ids[0]

# Load records
records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)

print(f"Fields in RecordSet {selected_record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
We demonstrate basic filtering and transformation using field `@id`s.

Steps:
- Filter records for a numeric field (e.g., molecular weight > threshold)
- Normalize the numeric field
- Optionally group by a categorical field

In [ ]:
# Inspect first few rows, pick a numeric field id to use for filtering/normalizing
numeric_field_ids = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_field_ids:
    numeric_field_id = numeric_field_ids[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No numeric fields detected; EDA steps may be limited.")

threshold = df[numeric_field_id].mean() if numeric_field_ids else None
if numeric_field_ids:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to find a categorical field (string/object)
    string_field_ids = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
    if string_field_ids:
        group_field_id = string_field_ids[0]
        print(f"\nAttempting to group by '{group_field_id}'")
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(grouped_df.head())
else:
    filtered_df = df.copy()

## 5. Visualization
Visualize the distribution of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_ids:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of '{numeric_field_id}' in {selected_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
else:
    print("No numeric fields to visualize.")

## 6. Conclusion
This notebook demonstrated how to access and process the FAIR\u00b2 protein mass spectrometry dataset using the `mlcroissant` library. We loaded metadata, explored available record sets and fields by `@id`, loaded records into DataFrames, performed simple numeric field filtering and normalization, and visualized data distributions.

**Next steps**: You can repeat the above process for other record sets or fields of interest. Refer to the printed field `@id`s to perform custom extraction and analyses tailored to your research questions.